In [3]:
#Plant disease detector
#This model is bulit by me for my Village farmers  who lose crops to diseases which they dont identify inculding my father

#Farmers sends a photo in whatsapp, This tool predicts the disease and sends the disease name and treatment for the respective disease
#It higlights the affected area through a heatmap so the farmers identify where the model is looking
#this notebook is only on this model it does not include whatsapp integration

In [4]:
!pip install -U kaggle split-folders gradio -q
#for file handling and data
import os
import pandas as pd
import numpy as np
import shutil

#for training  deep learning model
import tensorflow as tf
print("tensorflow imported sucessfully")

#for image processing and visualization
import cv2
import matplotlib.pyplot as plt
from PIL import Image

#for dataset splitting and app deployement
import splitfolders
print("splitfolders imported sucessfully")# just conforming colab has a litte problem with splitfolders
import gradio as gr

from google.colab import drive

#keras Stuff for building model
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2B0

#just checking versions so i know nothing broke
print("tensorflow:" , tf.__version__)
print("gradio", gr.__version__)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 20.9 MB/s eta 0:00:00
tensorflow imported sucessfully
splitfolders imported sucessfully
tensorflow: 2.20.0
gradio 6.20.0


In [5]:
drive.mount("/content/drive")

# making all my folders in my drive so i dont lose when my model when colab disconnects
project_dir="/content/drive/MyDrive/PlantDiseaseProject"

if not os.path.exists(project_dir):
    os.makedirs(project_dir)

#folders inside for models,outputs and samples
model_dir="/content/drive/MyDrive/PlantDiseaseProject/models"
output_dir = "/content/drive/MyDrive/PlantDiseaseProject/outputs"
sample_dir = "/content/drive/MyDrive/PlantDiseaseProject/samples"

os.makedirs(model_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(sample_dir, exist_ok=True)

# paths to save the model. FINAL one is after all epochs, BEST one is the best score during training
final_model_path = model_dir + "/efficientnetv2_plant_disease_model.keras"
best_model_path = model_dir + "/best_efficientnetv2_plant_disease_model.keras"

print("folders done")


Mounted at /content/drive
folders done


In [6]:
#setting up kaggle so i can download the dataset
#i need the kaggle.json api key file for this  i put mine in drive

kaggle_json_drive = "/content/drive/MyDrive/Kaggle/kaggle.json"
kaggle_json_local = "/root/.kaggle/kaggle.json"

os.makedirs("/root/.kaggle", exist_ok=True)

#copied the file from drive to the localkaggle folder so kaggle can find it
if os.path.exists(kaggle_json_drive):
  shutil.copy(kaggle_json_drive, kaggle_json_local)
  os.chmod(kaggle_json_local,0o600)# kaggle throws a warning if the file isnt 600 permissions, took me a while to figure that out

  print("kaggle.json loaded")
else:
  print("not found")


kaggle.json loaded


In [7]:
# downloading plant village dataset form kaggle
#only downloads when i need it(saves time when i rerun notebook)

raw_data_dir = "/content/plantvillage"
if not os.path.exists(raw_data_dir):
  #unzipping so it extracts automatically or i get a zip and have to unzip it myself
  !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/plantvillage --unzip
  print("dataset downloaded")
else:
  print("data set alrady there skipping downlaoding")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:19<00:00, 113MB/s]

dataset downloaded


In [8]:
#splitting the dataset into three folders train,val,test to prevent data leakage
# i am using colour images(there is also balck and white images and and segmented versions)

source_dir = "/content/plantvillage/plantvillage dataset/color"
split_data_dir = "/content/plantvillage_split"

if not os.path.exists(split_data_dir):
  splitfolders.ratio(
      source_dir,
      output=split_data_dir,
      seed=42,# seed 42 because i get exact result if i rerun anytime
      ratio=(0.70,0.15,0.15),# best to mantain balance
      group_prefix=None,
      move=False

  )
  print("data splitted")
else:
  print("data already splitted")

Copying files: 54305 files [00:12, 4327.47 files/s]

data splitted


In [9]:
img_size=224 #default for efficientnet
batch_size= 32 # Tried 64 but colab gave me OOM error

train_datagen= ImageDataGenerator( #arguementation for training so that it does not memorise image, it creates slightly changed versions of images
    rotation_range=20,
    horizontal_flip=True, # good for images
    zoom_range=0.2
)
# i didnt do arguemention because it prevents my validation accuracy drooping artificially
val_test_datagen=ImageDataGenerator()
#efficientnet has its own rescalling internally so i am not adding rescale=1./255 here

In [11]:
train_gen=train_datagen.flow_from_directory(
    f"{split_data_dir}/train",# if this path is slightly wrong everything breaks
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",# categorical because i have many classes,would have used binary if i have only two classes
    shuffle=True,#shuffeling because it does not learn the order of folders
    seed=42
)
val_gen= val_test_datagen.flow_from_directory(
    f"{split_data_dir}/val",
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False, # i didnt do true becuase because predictions wont line up with true labels in future
)
test_gen = val_test_datagen.flow_from_directory(
    f"{split_data_dir}/test",
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False # same here
)


Found 37997 images belonging to 38 classes.
Found 8129 images belonging to 38 classes.
Found 8179 images belonging to 38 classes.


In [19]:
# grabbing class count and names from the generator instead of hardcoding them (hardcoding = pain if dataset changes)
num_classes = len(train_gen.class_indices)
class_labels = list(train_gen.class_indices.keys())   # class_indices is a dict like {'Apple_scab': 0, ...}, keys are the names

print("number of classes:", num_classes)
print("train images:", train_gen.samples)
print("val images:", val_gen.samples)
print("test images:", test_gen.samples)
print(class_labels)

number of classes: 38
train images: 37997
val images: 8129
test images: 8179
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_lea

In [20]:
base_model=EfficientNetV2B0( #loading efficientnetv2 pretrained on imagenet this is the transferlearning part
    weights="imagenet", #imagenet  weights if i put none here i get random weights and transferlearning does not happen
    include_top=False, # false because i have only 38 classes not 1000
    include_preprocessing=True,# handles rescalling on its own thats why i skipped resclae=1./255 earlier
    input_shape=(img_size,img_size,3)# 3 RGB channels must match my image size
)

base_model.trainable=False # this freezes so the weights dont change during training


24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [29]:
#building my own model on top of frozen base
inputs = layers.Input(shape=(img_size, img_size, 3))
x = base_model(inputs, training=False)   # training is false becuase it keeps batchnorm layers frozen it matters when base is frozen
x = layers.GlobalAveragePooling2D()(x)   # this flattens the features i used this instead of Flatten because it cretes million of extra parameters and and overfits
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)   # dropout to prevent overfitting 0.3  drop 30 percent.too high 0.8 and it wont learn anything and 0.0 it overfits
outputs = layers.Dense(num_classes, activation="softmax", name="Plant_Disease_Output")(x)   # i used soft max because its 38 classes, would have used softmax if its only 2 classes

model = models.Model(inputs, outputs)

In [33]:
model.compile(
    optimizer="adam", #using this string becuase i forgot to imprt adam at starting.learning rate is default anyways0.001
    loss="categorical_crossentropy",# my labels are one-hot (like this [0,0,1,0..]) so i used this one. if labels were plain numbers like this one (0,1,2..) would have used  sparse_categorical_crossentropy instead
    metrics=["accuracy"]
)
model.summary()


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 7, 7, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_8      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Plant_Disease_Output (Dense)    │ (None, 38)             │        48,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,973,110 (22.79 MB)

 Trainable params: 51,238 (200.15 KB)

 Non-trainable params: 5,921,872 (22.59 MB)